In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")


In [ ]:
excel_file = 'online_retail_II.xlsx'
df1 = pd.read_excel(excel_file, sheet_name='Year 2009-2010')
df2 = pd.read_excel(excel_file, sheet_name='Year 2010-2011')
df = pd.concat([df1, df2], ignore_index=True)
df = df.dropna(subset=['Customer ID']) 
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
df['TotalPrice'] = df['Quantity'] * df['Price']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"Data loading and cleaning complete. Total valid transactions: {len(df)}")
df.head(3)

In [ ]:
split_date = pd.to_datetime('2010-12-01')

year1_customers = set(df[df['InvoiceDate'] < split_date]['Customer ID'])
year2_customers = set(df[df['InvoiceDate'] >= split_date]['Customer ID'])

churned_customers = year1_customers - year2_customers
churn_rate = len(churned_customers) / len(year1_customers) * 100

print(f"Total Year 1 Customers: {len(year1_customers)}")
print(f"Customers Lost in Year 2: {len(churned_customers)}")
print(f"Customer Churn Rate: {churn_rate:.2f}%")

In [ ]:
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'Invoice': 'nunique',                                    # Frequency
    'TotalPrice': 'sum'                                      # Monetary
}).reset_index()
rfm.columns = ['Customer ID', 'Recency', 'Frequency', 'Monetary']

rfm_clean = rfm[(rfm['Monetary'] < 50000) & (rfm['Frequency'] < 200)].copy()
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_clean[['Recency', 'Frequency', 'Monetary']])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_clean['Cluster'] = kmeans.fit_predict(rfm_scaled)

np.random.seed(42)
sample_indices = np.random.choice(len(rfm_scaled), size=min(5000, len(rfm_scaled)), replace=False)
sil_score = silhouette_score(rfm_scaled[sample_indices], rfm_clean['Cluster'].iloc[sample_indices])

print(f"K-Means Clustering Complete.")
print(f"Silhouette Score (Efficiency): {sil_score:.4f} (Score > 0.5 indicates well-defined clusters)")

In [ ]:
year1 = df[df['InvoiceDate'] < split_date]
y1_rfm = year1.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (split_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalPrice': 'sum',
    'Quantity': 'sum'
}).rename(columns={'InvoiceDate': 'Y1_Recency', 'Invoice': 'Y1_Frequency', 'TotalPrice': 'Y1_Monetary', 'Quantity': 'Y1_Items'})

# Target (y): How much they spent in Year 2
year2 = df[df['InvoiceDate'] >= split_date]
y2_spend = year2.groupby('Customer ID')['TotalPrice'].sum().rename('Y2_Spend')

# Merge and define X and y
clv_data = pd.merge(y1_rfm, y2_spend, left_index=True, right_index=True, how='inner')
X = clv_data[['Y1_Recency', 'Y1_Frequency', 'Y1_Monetary', 'Y1_Items']]
y = clv_data['Y2_Spend']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Random Forest Training Complete.")
print(f"Model R-Squared: {r2:.2f} (Explains {r2*100:.0f}% of the variance in future spend)")
print(f"Model RMSE: £{rmse:.2f}")

In [ ]:
# Cell 5: Predicting Customer Lifetime Value (CLV)
# Features (X): What the customer did in Year 1
year1 = df[df['InvoiceDate'] < split_date]
y1_rfm = year1.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (split_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalPrice': 'sum',
    'Quantity': 'sum'
}).rename(columns={'InvoiceDate': 'Y1_Recency', 'Invoice': 'Y1_Frequency', 'TotalPrice': 'Y1_Monetary', 'Quantity': 'Y1_Items'})

# Target (y): How much they spent in Year 2
year2 = df[df['InvoiceDate'] >= split_date]
y2_spend = year2.groupby('Customer ID')['TotalPrice'].sum().rename('Y2_Spend')

# Merge and define X and y
clv_data = pd.merge(y1_rfm, y2_spend, left_index=True, right_index=True, how='inner')
X = clv_data[['Y1_Recency', 'Y1_Frequency', 'Y1_Monetary', 'Y1_Items']]
y = clv_data['Y2_Spend']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Random Forest Training Complete.")
print(f"Model R-Squared: {r2:.2f} (Explains {r2*100:.0f}% of the variance in future spend)")
print(f"Model RMSE: £{rmse:.2f}")

In [ ]:
# Cell 6: GTM Visualizations & Dashboard Output
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Monetary Profiles of our Clusters
sns.boxplot(x='Cluster', y='Monetary', data=rfm_clean[rfm_clean['Monetary'] < 5000], ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('Cluster Profiles: Monetary Spend (Capped at £5000)', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Total Spend (£)', fontsize=12)
axes[0, 0].set_xlabel('GTM Customer Segment (0-3)', fontsize=12)

# Plot 2: Recency Profiles of our Clusters
sns.boxplot(x='Cluster', y='Recency', data=rfm_clean, ax=axes[0, 1], palette='viridis')
axes[0, 1].set_title('Cluster Profiles: Recency', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Days Since Last Purchase', fontsize=12)
axes[0, 1].set_xlabel('GTM Customer Segment (0-3)', fontsize=12)

# Plot 3: Predictive ML Performance
sns.scatterplot(x=y_test, y=y_pred, ax=axes[1, 0], alpha=0.6, color='#2ca02c')
axes[1, 0].plot([0, 15000], [0, 15000], 'r--', lw=2) # Ideal accuracy line
axes[1, 0].set_xlim(0, 15000)
axes[1, 0].set_ylim(0, 15000)
axes[1, 0].set_title(f'ML Prediction: Actual vs Predicted CLV (R² = {r2:.2f})', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Actual Spend in Year 2 (£)', fontsize=12)
axes[1, 0].set_ylabel('Predicted Spend by Random Forest (£)', fontsize=12)

# Plot 4: Churn Analysis
axes[1, 1].pie([len(year1_customers) - len(churned_customers), len(churned_customers)], 
               labels=['Retained', 'Churned'], autopct='%1.1f%%', colors=['#4c72b0', '#c44e52'], 
               startangle=140, explode=(0, 0.1), textprops={'fontsize': 12})
axes[1, 1].set_title(f'Customer Churn Rate (Year 1 → Year 2)', fontsize=14, fontweight='bold')

plt.tight_layout(pad=3.0)
plt.show()